In [5]:
import os
import re
import subprocess
from collections import deque

the_directory = 'output/demo'
md_file    = 'the_start_markdown.txt'
src_file   = 'the_functions_all.txt'
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# (reuse your find_function_native / find_function_source_in_file here)
def find_function_source_in_file(path: str, function_name: str) -> str | None:
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')
    def_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*(?:\s*\*+)?\s+)+
            \**\s*
            {re.escape(function_name)}\s*\(
        ''',
        re.VERBOSE
    )
    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None
    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False
        for raw in f:
            line = raw
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue
            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]
            clean = line.split('//',1)[0]
            if clean.rstrip().endswith('\\'):
                continue
            if not collecting:
                if sig_re.search(clean) and def_re.match(clean):
                    sig_raw = [raw]
                    sig_clean = [clean]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        nxt = nxt_raw
                        if in_block:
                            end = nxt.find('*/')
                            if end >= 0:
                                in_block = False
                                nxt = nxt[end+2:]
                            else:
                                continue
                        st = nxt.find('/*')
                        if st >= 0:
                            ed = nxt.find('*/', st+2)
                            if ed >= 0:
                                nxt = nxt[:st] + nxt[ed+2:]
                            else:
                                in_block = True
                                nxt = nxt[:st]
                        nxt_clean = nxt.split('//',1)[0]
                        sig_raw.append(nxt_raw)
                        sig_clean.append(nxt_clean)
                        bal += nxt_clean.count('(') - nxt_clean.count(')')
                    if ''.join(sig_clean).strip().endswith(';'):
                        continue
                    buffer = sig_raw.copy()
                    if '{' in sig_clean[-1]:
                        collecting = True
                        brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                    else:
                        for body_raw in f:
                            body = body_raw
                            if in_block:
                                end = body.find('*/')
                                if end >= 0:
                                    in_block = False
                                    body = body[end+2:]
                                else:
                                    continue
                            st = body.find('/*')
                            if st >= 0:
                                ed = body.find('*/', st+2)
                                if ed >= 0:
                                    body = body[:st] + body[ed+2:]
                                else:
                                    in_block = True
                                    body = body[:st]
                            body_clean = body.split('//',1)[0]
                            buffer.append(body_raw)
                            if '{' in body_clean:
                                collecting = True
                                brace_count = (
                                    body_clean.count('{') -
                                    body_clean.count('}')
                                )
                                break
                    if not collecting:
                        buffer = []
            else:
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]
                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)
    return None

def find_function_native(root_dir: str, function_name: str) -> str | None:
    pat = rf'{re.escape(function_name)}[[:space:]]*\('
    try:
        out = subprocess.check_output([
            'grep', '-RlE',
            '--include=*.c', pat, root_dir
        ], text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return None
    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src
    return None
# 1) Parse markdown jadi list pasangan parent→child
pairs = []
stack = []
with open(f'{the_directory}/{md_file}') as f:
    for line in f:
        if not line.strip(): continue
        indent = len(line) - len(line.lstrip(' '))
        name   = line.lstrip(' -').strip()
        while stack and stack[-1][0] >= indent:
            stack.pop()
        if stack:
            pairs.append((stack[-1][1], name))
        stack.append((indent, name))
# Debug: cek apakah kita benar-benar punya pasangan
#print(f"[DEBUG] Ketemu {len(pairs)} pairs:")
#for p, c in pairs:
    #print(f"  - {p} → {c}")
# 2) Load blok sumber yang ada
with open(f"{the_directory}/{src_file}") as f:
    content = f.read()
pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?=XXXTHISENDSHEREXXX)"
function_sources = dict(re.findall(pattern, content, flags=re.DOTALL))

# 3) Tambahkan fungsi dari markdown jika belum ada
all_funcs = {fn for p in pairs for fn in p}
for fn in all_funcs:
    if fn not in function_sources:
        print('a')
        src = find_function_native(search_dir, fn)
        print('b')
        if src:
            function_sources[fn] = src

# 4) Closure: tambahkan semua callee yang belum di-parse
while True:
    added = False
    for fn, src in list(function_sources.items()):
        for callee in set(re.findall(r'\b([A-Za-z_]\w*)\b', src)):
            if callee not in function_sources:
                src2 = find_function_native(search_dir, callee)
                if src2:
                    function_sources[callee] = src2
                    added = True
    if not added:
        break

# 5) Build adjacency list
adj = {fn: set() for fn in function_sources}
for fn, src in function_sources.items():
    for callee in function_sources:
        if callee != fn and re.search(rf'\b{re.escape(callee)}\b', src):
            adj[fn].add(callee)
# Debug: cek adjacency untuk fsnotify_find_mark
#print("[DEBUG] adj[fsnotify_find_mark] =", adj.get("fsnotify_find_mark"))
# 6) BFS dan print langsung setiap chain
for parent, child in pairs:
    print(f"[DEBUG] Proses: {parent} → {child}")
    if parent not in adj or child not in function_sources:
        print(f"{parent} -> {child} : (no source)")
        continue
    if child in adj[parent]:
        print(f"{parent}, {child}")
        continue

    visited = {parent}
    queue   = deque([[parent]])
    found   = None
    while queue and not found:
        path = queue.popleft()
        for nb in adj[path[-1]]:
            if nb in visited:
                continue
            visited.add(nb)
            new_path = path + [nb]
            if nb == child:
                found = new_path
                break
            queue.append(new_path)
    if found:
        print(', '.join(found))
    else:
        print(f"{parent} -> {child} : (no path)")
print("Selesai.")

a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a
b
a


KeyboardInterrupt: 